In [ ]:
import sys
sys.executable

In [ ]:
!python --version

### Gnina Workshop

https://colab.research.google.com/drive/1GXmk1v8C-c4UtyKFqIm9HnsrVYH0pI-c#scrollTo=BrgAAkFIEh5u


### Youtube

https://www.youtube.com/watch?view=MG3Srzi5kZ0

### pdbqt

A .pdbqt file is a specialized file format for molecular modeling, used primarily with the AutoDock suite for virtual screening and docking. It's an enhanced version of a standard PDB file that includes additional information crucial for docking calculations, specifically polar hydrogens, partial charges (Q), and AutoDock atom types (T). These additions are necessary for accurate calculations of non-covalent interactions during the docking process

In [ ]:
import os, sys
# sys.path.insert(1 '../src/')

import subprocess
# from IPython.display import display

from pathlib import Path
ROOT = Path().resolve().parent
SRC = os.path.join(ROOT, "src")

if str(SRC) not in sys.path:
    sys.path.append(str(SRC))

print("ROOT:", ROOT)
print("SRC added:", SRC)

# first Basica next pdb_lib
from docking_gnina.Basic import *
from docking_gnina.pdb_lib import *

### PDB instantiation

In [ ]:
ROOT

In [ ]:
os.listdir(".")

In [ ]:
root_data= os.path.join(ROOT, 'data') 
pdb = PDB(root_data=root_data)

In [ ]:
pdb.root_data

In [ ]:
os.listdir(root_data)

In [ ]:
# !uv add openmm --reinstall

In [ ]:
# from openmm import unit as openmm_unit

1*openmm_unit.picosecond

In [ ]:
!uv add openmm

### Testing CUDA

In [ ]:
!nvidia-smi

In [ ]:
import torch
# import torch.nn as nn
# import torch.nn.functional as F
# import torch.optim as optim
# from torch.optim.lr_scheduler import ExponentialLR
# from torch.utils.data import Dataset, DataLoader, Subset

print(torch.cuda.is_available())

if torch.cuda.is_available():
    print(">> device counts:", torch.cuda.device_count())
    print(">> current device:", torch.cuda.current_device())
    print(">> device name:", torch.cuda.get_device_name(0))

torch.__version__

In [ ]:
root_result = os.path.join(root_data, 'docking_files')
os.path.exists(root_result), root_result

#### Protein:

https://www.rcsb.org/structure/

> choose a protein receptor

#### Ligand:

https://pubchem.ncbi.nlm.nih.gov/compound/

> choose a ligand

In [ ]:
root_data

In [ ]:
fname_protein_receptor = 'xxxx.pdb'

fname_ligand = 'yyyy.sdf'

receptor_file = os.path.join(root_data, fname_protein_receptor)
ligand_file = os.path.join(root_data, fname_ligand)

fname_result = fname_protein_receptor.replace('.pdb','') + '_result.pdb'
output_file = os.path.join(root_result, fname_result)

In [ ]:
os.path.exists(receptor_file), receptor_file

In [ ]:
os.path.exists(ligand_file), ligand_file

### Prepare ligand: add hidrogens

In [ ]:
# !obabel ../data/Otx008.sdf -O ../data/Otx008_prepared.sdf -h --gen3d

In [ ]:
fname_ligand_preared = 'Otx008_prepared.sdf'
ligand_file_prep = os.path.join(root_data, fname_ligand_preared)

os.path.exists(ligand_file_prep), fname_ligand_preared

In [ ]:
os.path.exists(output_file), output_file

In [ ]:
!gnina --help

In [ ]:
!gnina --version

### Running CUDA

In [ ]:
if not os.path.exists(root_result):
    create_dir(root_result)

In [ ]:
result_files = os.listdir(root_result)
len(result_files)

### Visualize prepared ligand

In [ ]:
ligand_file_prep

In [ ]:
view = pdb.visualize_a_ligand(ligand_file=ligand_file_prep, 
		ligand_color=None, ligand_radius=0.15)

if view: 
    view.zoomTo()
    view.show()

In [ ]:
result_files, fname_protein_receptor

In [ ]:
%time

if fname_result not in result_files:

    cmd = [
        "gnina",
        "-r", receptor_file,
        "-l", ligand_file_prep,
        "-o", output_file,
        "--seed", "0",
        "--autobox_ligand", receptor_file,
        "--exhaustiveness", "16"
    ]
    
    subprocess.run(cmd, check=True)
else:
    print("Already processed", os.path.exists(output_file))

#  
    


#### ⚠️ 1️⃣ “initial pose not within box”

11953346 | pose 0 | initial pose not within box


Isso significa:

> 👉 A pose inicial do ligante estava fora da caixa de busca (search box).  
> 👉 O GNINA reposicionou automaticamente durante o docking.  


#### Affinity (Vina-like)

Score tipo Vina (kcal/mol).

Melhor modelo: -7.59 kcal/mol

Para referência aproximada:
  - -5 a -6 → fraco
  - -7 a -8 → moderado
  - -9 ou menor → forte

Seu resultado: moderado baixo.


#### 🚨 Intramolecular energy 

Intramolecular energy mede tensão interna do ligante.

-3 a -4 kcal/mol  ✅ perfeito

Isso indica:

> ✔ Ligante bem preparado  
> ✔ Geometria correta  
> ✔ Sem distorção artificial  

Excelente sinal.


#### 🚨 CNN pose score tem que ser >= 0.3

> ✔ 0.0 → ruim  
> ✔ 0.3–0.6 → razoável  
> ✔ 0.7 → muito bom  

#### 🚨 CNN affinity 7.359

Importante:

> ✔ CNN affinity não é negativo como Vina.  
> ✔ Ele é uma estimativa aprendida por rede neural.  
> ✔ Valores ~7 sugerem afinidade moderada.  


#### 🎯 Conclusão técnica

Seu docking agora parece:

> ✔ Numericamente consistente  
> ✔ Geometricamente plausível  
> ✔ CNN funcionando  
> ✔ GPU provavelmente ativa  

Isso agora é resultado válido para análise.


### Visualize Docking

In [ ]:
output_file

In [ ]:
view = pdb.visualize_receptor_ligand(receptor_file=receptor_file, ligand_file="", 
        receptor_radius=0.1, ligand_color='green', ligand_radius=0.15,
		cognate_file=output_file, cognate_color="yellow", cognate_radius=.25,
        model=1, zoom_to_model=1, rotate=270)

if view: 
    view.zoomTo()
    view.show()

### Visualize Receptor

In [ ]:
# only receptor
view = pdb.visualize_receptor_ligand(receptor_file=receptor_file, ligand_file="", 
        receptor_radius=0.1, ligand_color='green', ligand_radius=0.15,
		cognate_file="", cognate_color="yellow", cognate_radius=.25,
        model=1, zoom_to_model=-1, rotate=180)

if view: 
    view.zoomTo()
    view.show()

In [ ]:
receptor_file, ligand_file_prep, output_file

### visualize_poses

In [ ]:
for the_content in range(9):
    view = pdb.visualize_one_pose(
        receptor_file=receptor_file,
        ligand_file=output_file, ligand_color_scheme='yellow', the_content=the_content,
        width=800, height=600)

    if view:
        view.rotate(270)
        view.show()

### Chose a pose

In [ ]:
the_content = 2

view = pdb.visualize_one_pose(
    receptor_file=receptor_file,
    ligand_file=output_file, ligand_color_scheme='yellow', the_content=the_content,
    width=800, height=600)

if view:
    view.rotate(270)
    view.show()


### Animate

In [ ]:
view = pdb.visualize_poses(
    protein_file=receptor_file,
    pose_file=output_file,
    cognate_file="",
    animate=True,
)  # Change to True to see an animation of all of the poses

if view:
    view.show()

In [ ]:
# sudo apt-get install openbabel


### obrms: Computes the heavy-atom RMSD of identical compound structures.

Isso usa o obrms do Open Babel.

O problema é que o obrms:

> ❌ NÃO alinha automaticamente moléculas  
> ❌ Exige mesma orientação e mesmo sistema de coordenadas  
> ❌ Pode comparar moléculas não sobrepostas  


In [ ]:
!obrms

In [ ]:
# !obrms --firstonly ../../colaboracoes/cheminformatics/3ERK_lig.pdb ../../colaboracoes/cheminformatics/docking_files/docking_results/docked.sdf

In [ ]:
receptor_file, ligand_file_prep, output_file

### Not aligned

In [ ]:
cmd_line = f"obrms --firstonly {ligand_file_prep} {output_file}"
os.system(cmd_line)

🚨 Esses RMSDs estão completamente impossíveis.

Valores > 20 Å --> problemas

> não são “dockings ruins”.  
> São cálculo errado de RMSD.  
  

🎯 Referência real

Em redocking:

> < 2 Å → excelente  
> 2–3 Å → aceitável  
> \> 4 Å → ruim  

200 Å significa:

👉 As moléculas não estão sendo alinhadas
👉 Ou você está comparando coisas erradas

🔎 Causa mais provável

Você está calculando RMSD entre:

> Moléculas em sistemas de coordenadas diferentes  
> Ou sem alinhamento prévio  
> Ou com número de átomos diferente  
> Ou incluindo proteína inteira  